# ML02 · 線性迴歸與梯度下降

用最小的線性模型，把「定義模型、量誤差、靠梯度逐步修正參數」這個訓練迴圈完整走一遍，建立後面所有神經網路的縮影。

## 學習目標
- 理解線性模型（linear model，y = wx + b）的意義，知道參數 w 與 b 各自控制什麼。
- 能用均方誤差（mean squared error，MSE）量化預測與真實值的差距，並理解它是要被降低的「損失（loss）」。
- 建立梯度下降（gradient descent）的直覺：為什麼順著梯度反方向走可以降低損失。
- 理解學習率（learning rate）對收斂的影響，能說明太大與太小各會發生什麼。
- 能手刻一次完整的參數更新迴圈（前向計算、算損失、算梯度、更新參數）。
- 能用 sklearn 的 LinearRegression 對照驗證手刻結果，確認方向正確。

In [ ]:
# 概念：畫圖前先設定 matplotlib，讓中文標題不會變成方框
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

# 依序試常見中文字型，挑到哪個系統有就用哪個（不同作業系統字型名稱不同）
matplotlib.rcParams["font.sans-serif"] = ["Microsoft JhengHei", "PingFang TC", "Heiti TC", "SimHei", "Arial Unicode MS", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False   # 避免負號顯示成亂碼

print("matplotlib 版本:", matplotlib.__version__)
print("numpy 版本:", np.__version__)

## 線性模型是什麼

線性模型（linear model）就是「一條可以調整的直線」：y = wx + b。
- 權重（weight，w）是斜率，控制 x 每增加一單位、y 跟著變多少。
- 偏置（bias，b）是截距，控制這條線整體往上或往下平移。

為什麼先學它：把模型看成「一組參數（w, b）」，之後所有的學習其實都是在調這組參數，讓預測（prediction）更貼近真實值。線性模型是這個觀念最小、最乾淨的起點。

形狀：y_pred = w * x + b，其中 x 是一批輸入，w、b 是兩個純量。

In [ ]:
# 概念：把線性模型看成一條由參數 (w, b) 決定的直線，手動試幾組參數看擬合好壞
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)   # 固定亂數種子，讓每次執行的雜訊一致，方便對照

# 造一批帶雜訊的 (x, y) 點：仿真一條真實關係 y = 2x + 5，再加上隨機雜訊
x = np.linspace(0, 10, 30)                       # 0 到 10 取 30 個等距點當輸入
true_w, true_b = 2.0, 5.0                         # 假設背後真正的斜率與截距
noise = rng.normal(0, 2.0, size=x.shape)          # 常態雜訊，模擬量測誤差
y = true_w * x + true_b + noise

def predict(x, w, b):
    # 線性模型的前向計算：一次算出所有點的預測值（向量化，不用迴圈）
    return w * x + b

# 手動試兩組參數：一組刻意偏掉、一組接近真值
guess_bad = predict(x, w=1.0, b=0.0)
guess_good = predict(x, w=2.0, b=5.0)

plt.figure(figsize=(6, 4))
plt.scatter(x, y, label="資料點", color="gray")
plt.plot(x, guess_bad, label="w=1, b=0（偏掉）", color="orange")
plt.plot(x, guess_good, label="w=2, b=5（較貼）", color="green")
plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.title("不同參數畫出不同的線")
plt.show()

print("前 3 點的真實 y:", np.round(y[:3], 2))
print("w=2,b=5 的預測:", np.round(guess_good[:3], 2))

## 用 MSE 量誤差

肉眼判斷「哪條線比較貼」不夠客觀，需要一個數字代表「整體有多錯」。這個數字就是損失（loss）。

均方誤差（mean squared error，MSE）是最常用的回歸損失：把每個點的殘差（residual，預測值減真實值）平方後取平均。平方讓正負誤差不會互相抵消，也讓大誤差被放大。MSE 越小代表線越貼，這就是我們要最小化的目標。

公式：MSE = 平均（(y_pred − y_true)²）。

In [ ]:
# 概念：寫一個 mse 函式把整體誤差濃縮成一個數字，比較不同參數的好壞
import numpy as np

rng = np.random.default_rng(0)
x = np.linspace(0, 10, 30)
y = 2.0 * x + 5.0 + rng.normal(0, 2.0, size=x.shape)   # 與上一節相同的自造資料

def mse(y_true, y_pred):
    residual = y_pred - y_true     # 殘差：每個點預測偏離真實多少
    return np.mean(residual ** 2)  # 平方後取平均，得到單一損失數字

def predict(x, w, b):
    return w * x + b

# 比較三組參數的 MSE：越接近真值 (w=2, b=5) 應該越小
for w, b in [(1.0, 0.0), (2.0, 0.0), (2.0, 5.0)]:
    loss = mse(y, predict(x, w, b))
    print(f"w={w}, b={b}  ->  MSE = {loss:.3f}")

## 梯度下降的直覺

我們想找一組 (w, b) 讓 MSE 最小，但不可能無限地亂試。梯度下降（gradient descent）給出有方向的找法。

把損失想成一座山，山的高度就是 MSE，腳下的位置就是當前參數。梯度（gradient）指向「上坡最陡」的方向，所以往梯度的反方向走一小步，損失就會下降，這就是「下山」的比喻。整個損失隨參數變化的形狀稱為損失曲面（loss surface）。

關鍵直覺：對 w 的梯度若為正，代表往 w 變大會讓損失上升，所以該把 w 調小；為負則反之。

In [ ]:
# 概念：固定 b、只掃描 w，畫出損失的碗形曲線並標出當前點與下山方向
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
x = np.linspace(0, 10, 30)
y = 2.0 * x + 5.0 + rng.normal(0, 2.0, size=x.shape)

def mse(y_true, y_pred):
    return np.mean((y_pred - y_true) ** 2)

fixed_b = 5.0
w_grid = np.linspace(-1, 5, 100)                          # 掃描一段 w 值
losses = [mse(y, w * x + fixed_b) for w in w_grid]        # 每個 w 對應一個損失，組成碗形曲線

current_w = 0.5                                            # 假設目前站在這個 w
# 對 w 的梯度（MSE 對 w 微分後的解析式）：方向告訴我們該往哪走
current_grad = np.mean(2 * (current_w * x + fixed_b - y) * x)
current_loss = mse(y, current_w * x + fixed_b)

plt.figure(figsize=(6, 4))
plt.plot(w_grid, losses, label="損失隨 w 變化")
plt.scatter([current_w], [current_loss], color="red", zorder=5, label="目前位置")
# 梯度為正代表上坡在右邊，下山要往左（w 變小）；畫一個指向下坡的箭頭示意
plt.annotate("", xy=(current_w - 0.8, current_loss), xytext=(current_w, current_loss),
             arrowprops=dict(arrowstyle="->", color="red"))
plt.xlabel("w"); plt.ylabel("MSE"); plt.legend(); plt.title("損失曲面（固定 b 只看 w）")
plt.show()

print(f"目前 w={current_w} 的梯度 = {current_grad:.2f}（正數代表該把 w 調小才會下山）")

## 學習率與收斂

梯度只給「方向」，但每一步要走多大由學習率（learning rate）決定，這個步幅也稱為步長（step size）。

口訣是「方向由梯度給，幅度由學習率給」。學習率太小，每步挪一點點，要很多步才到谷底（收斂 convergence 很慢）；太大則可能一步跨過谷底、在兩側來回彈，甚至越跑越遠而發散（divergence）。挑一個適中的學習率，是訓練能不能成功的關鍵。

In [ ]:
# 概念：用同一筆資料跑三種學習率，比較損失隨迭代下降的曲線（收斂 vs 發散）
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
x = np.linspace(0, 10, 30)
y = 2.0 * x + 5.0 + rng.normal(0, 2.0, size=x.shape)
n = len(x)

def run(lr, steps=40):
    w, b = 0.0, 0.0                              # 都從同一個起點出發，公平比較
    history = []
    for _ in range(steps):
        y_pred = w * x + b
        history.append(np.mean((y_pred - y) ** 2))
        # 梯度：MSE 分別對 w、b 微分得到的解析式
        grad_w = (2 / n) * np.sum((y_pred - y) * x)
        grad_b = (2 / n) * np.sum(y_pred - y)
        w -= lr * grad_w                          # 往梯度反方向走，步幅由 lr 控制
        b -= lr * grad_b
    return history

plt.figure(figsize=(6, 4))
# 注意：過大的學習率會讓損失不減反增，曲線往上甚至爆掉
for lr, name in [(0.0005, "過小 lr=0.0005"), (0.01, "適中 lr=0.01"), (0.025, "過大 lr=0.025")]:
    plt.plot(run(lr), label=name)
plt.xlabel("迭代次數"); plt.ylabel("MSE"); plt.legend(); plt.title("不同學習率的下降曲線")
plt.yscale("log")   # 技巧：用對數刻度，讓收斂與發散的差距在同一張圖看得清楚
plt.show()

## 手刻訓練迴圈

現在把前面的零件串成一個完整的訓練迴圈。每一輪（迭代 iteration，掃過全部資料一次也稱為一個 epoch）做四件事：
1. 前向計算（forward pass）：用目前的 w、b 算出預測。
2. 算損失：用 MSE 看現在有多錯。
3. 算梯度：得到每個參數該往哪調。
4. 參數更新（parameter update）：往梯度反方向各走一步。

重複數十次直到損失幾乎不再下降，就近似收斂。這個迴圈就是後面所有神經網路訓練的縮影，差別只在模型更大、參數更多。

In [ ]:
# 概念：把前向、損失、梯度、更新串成一個 for 迴圈，從隨機起點迭代出最終 w、b
import numpy as np

rng = np.random.default_rng(0)
x = np.linspace(0, 10, 30)
y = 2.0 * x + 5.0 + rng.normal(0, 2.0, size=x.shape)   # 真值 w=2, b=5，看迴圈能不能逼近
n = len(x)

w, b = rng.normal(0, 1), rng.normal(0, 1)   # 隨機初始參數，模擬「一無所知的起點」
lr = 0.01
epochs = 60

for epoch in range(epochs):
    y_pred = w * x + b                         # 1. 前向計算
    loss = np.mean((y_pred - y) ** 2)          # 2. 算損失
    grad_w = (2 / n) * np.sum((y_pred - y) * x)  # 3. 算梯度
    grad_b = (2 / n) * np.sum(y_pred - y)
    w -= lr * grad_w                           # 4. 更新參數（同時更新 w 與 b）
    b -= lr * grad_b
    if epoch % 10 == 0 or epoch == epochs - 1:   # 每隔幾輪印一次，觀察損失逐步下降
        print(f"epoch {epoch:2d}  loss={loss:7.3f}  w={w:.3f}  b={b:.3f}")

print("---")
print(f"最終參數: w={w:.3f}, b={b:.3f}（真值為 w=2.0, b=5.0）")

## 用 sklearn 對照驗證

手刻迴圈得到一組 w、b，但怎麼知道它對不對？用成熟工具求出「標準答案」來對照。

scikit-learn 的 LinearRegression 會用解析解（最小平方法）直接算出最佳參數：呼叫 fit 擬合資料後，用 coef_ 取得係數（斜率）、用 intercept_ 取得截距。它背後做的事與手刻迴圈一致，只是把流程封裝起來。若手刻結果與它接近，就代表方向正確。

In [ ]:
# 概念：用 sklearn LinearRegression 求標準答案，與手刻迴圈的參數並排比較
import numpy as np
from sklearn.linear_model import LinearRegression

rng = np.random.default_rng(0)
x = np.linspace(0, 10, 30)
y = 2.0 * x + 5.0 + rng.normal(0, 2.0, size=x.shape)
n = len(x)

# 先重跑一次手刻迴圈，拿到手刻的 w、b
w, b = 0.0, 0.0
for _ in range(2000):
    y_pred = w * x + b
    w -= 0.01 * (2 / n) * np.sum((y_pred - y) * x)
    b -= 0.01 * (2 / n) * np.sum(y_pred - y)

# 注意：sklearn 的 fit 要求特徵是二維 (樣本數, 特徵數)，所以把一維 x 變成欄向量
model = LinearRegression()
model.fit(x.reshape(-1, 1), y)

print(f"手刻迴圈   : w={w:.4f}, b={b:.4f}")
print(f"sklearn   : w={model.coef_[0]:.4f}, b={model.intercept_:.4f}")
print(f"兩者差距   : dw={abs(w - model.coef_[0]):.4f}, db={abs(b - model.intercept_):.4f}")

## 練習

以下三題由淺入深，皆為複合型但簡短。資料請自己用 numpy 造（仿真即可），完成後對照「驗收」確認輸出。

In [ ]:
# TODO 1 ·（簡單）用一個特徵預測房價（整合：線性模型 + MSE）
#   情境：自造一批公寓資料，用「樓地板面積」單一特徵預測房價（仿真線性加雜訊）。
#   要求：
#     1. 用 numpy 自造 (面積, 房價) 資料，並手選一組 w、b 用 predict 畫出預測線。
#     2. 寫 mse 函式，調整 w、b 試著把 MSE 壓到更低，記下最好的一組並印出。
#   驗收：應該看到 MSE 數值隨著參數調整而下降，且預測線明顯更貼近資料點。
# TODO: 在這裡完成


In [ ]:
# TODO 2 ·（中間）手刻梯度下降擬合容積率（整合：MSE + 梯度下降 + 學習率 + 訓練迴圈）
#   情境：自造一批基地資料，用「建蔽率」預測「容積率」（仿真線性關係）。
#   要求：
#     1. 從隨機初始 w、b 開始，寫一個梯度下降迴圈逐步更新參數。
#     2. 記錄每次迭代的 MSE，畫出損失下降曲線。
#     3. 換一個明顯過大的學習率重跑，觀察曲線變化。
#   驗收：應該看到適中學習率下損失平滑下降並收斂，而過大學習率下損失震盪或發散。
# TODO: 在這裡完成


In [ ]:
# TODO 3 ·（稍難）手刻與 sklearn 雙軌驗證並調學習率（整合：訓練迴圈 + 學習率 + sklearn 對照 + 結果分析）
#   情境：自造一批街廓資料，用「臨路寬度」預測「平均樓高」（仿真線性加雜訊）。
#   要求：
#     1. 手刻梯度下降迴圈求 w、b，並嘗試至少三種學習率挑出最佳收斂者。
#     2. 用 sklearn LinearRegression 對同一資料求標準答案（記得把 x reshape 成二維）。
#     3. 比較手刻與 sklearn 的參數差距，並用註解簡述：若差距偏大，可能是學習率或迭代次數哪裡需要調整。
#   驗收：應該看到手刻參數與 sklearn 結果接近，且能說出讓兩者更一致的調整方向。
# TODO: 在這裡完成


## 小結

- 線性模型是一條由參數 (w, b) 決定的直線：w 是斜率、b 是截距；學習的本質就是調這組參數。
- MSE 把每點殘差平方再平均，用一個數字代表整體誤差，是要被最小化的損失。
- 梯度下降把損失當成一座山，往梯度反方向走一小步就會下山，逐步逼近最低點。
- 學習率決定每步的幅度：太小收斂慢、太大會震盪甚至發散；方向由梯度給、幅度由學習率給。
- 手刻訓練迴圈就是「前向計算、算損失、算梯度、更新參數」重複數十次，這正是神經網路訓練的縮影。
- 用 sklearn LinearRegression 求出的係數與截距可當標準答案，驗證手刻結果方向正確。